In [1]:
import mne
import numpy as np

from pathlib import Path

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import iirnotch
from scipy.signal import welch

from scipy.stats import skew
from scipy.stats import kurtosis

In [2]:
pairs = [
    ('FC5','FC6'),
    ('FC3','FC4'),
    ('FC1','FC2'),

    ('C5','C6'),
    ('C3','C4'),
    ('C1','C2'),

    ('CP5','CP6'),
    ('CP3','CP4'),
    ('CP1','CP2'),

    ('FP1','FP2'),

    ('AF7','AF8'),
    ('AF3','AF4'),

    ('F7','F8'),
    ('F5','F6'),
    ('F3','F4'),
    ('F1','F2'),

    ('FT7','FT8'),

    ('T7','T8'),
    ('T9','T10'),

    ('TP7','TP8'),

    ('P7','P8'),
    ('P5','P6'),
    ('P3','P4'),
    ('P1','P2'),

    ('PO7','PO8'),
    ('PO3','PO4'),

    ('O1','O2')
]

print("Number of pairs =", len(pairs))

Number of pairs = 27


In [3]:
def create_differential_channels(raw, pairs):

    # Clean channel names
    clean_names = {
        ch: ch.replace(".", "").upper()
        for ch in raw.ch_names
    }

    raw.rename_channels(clean_names)

    # Get EEG data
    data = raw.get_data()

    # Create fresh channel index dictionary
    ch_idx = {
        ch: i
        for i, ch in enumerate(raw.ch_names)
    }

    diff_data = []

    for left, right in pairs:

        left_idx = ch_idx[left]
        right_idx = ch_idx[right]

        diff_signal = data[left_idx] - data[right_idx]

        diff_data.append(diff_signal)

    diff_data = np.array(diff_data)

    diff_names = [
        f"{left}-{right}"
        for left, right in pairs
    ]

    return diff_data, diff_names

In [4]:
def apply_notch_filter(data, fs=160, freq=50, Q=30):

    b, a = iirnotch(
        w0=freq,
        Q=Q,
        fs=fs
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [5]:
def apply_bandpass_filter(
    data,
    fs=160,
    lowcut=0.5,
    highcut=70,
    order=4
):

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=1
    )

    return filtered

In [6]:
def minmax_normalize(data):

    data_min = np.min(
        data,
        axis=1,
        keepdims=True
    )

    data_max = np.max(
        data,
        axis=1,
        keepdims=True
    )

    normalized = (
        data - data_min
    ) / (
        data_max - data_min
    )

    return normalized

In [7]:
def extract_11_features(signal, fs=160):

    features = []

    # 1 Mean
    features.append(np.mean(signal))

    # 2 Variance
    features.append(np.var(signal))

    # 3 Skewness
    features.append(skew(signal))

    # 4 Kurtosis
    features.append(kurtosis(signal))

    # 5 Zero Crossing Count
    zc = np.sum(
        np.diff(
            np.sign(signal)
        ) != 0
    )

    features.append(zc)

    # 6 Area
    features.append(
        np.trapezoid(np.abs(signal))
    )

    # 7 Range
    features.append(
        np.max(signal) - np.min(signal)
    )

    # PSD
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=len(signal)
    )

    # 8 Delta
    delta = np.sum(
        psd[
            (freqs >= 0.5) &
            (freqs < 4)
        ]
    )

    # 9 Theta
    theta = np.sum(
        psd[
            (freqs >= 4) &
            (freqs < 8)
        ]
    )

    # 10 Alpha
    alpha = np.sum(
        psd[
            (freqs >= 8) &
            (freqs < 12)
        ]
    )

    # 11 Beta
    beta = np.sum(
        psd[
            (freqs >= 12) &
            (freqs < 30)
        ]
    )

    features.extend([
        delta,
        theta,
        alpha,
        beta
    ])

    return np.array(features)

In [8]:
def process_edf_file(edf_path):

    # --------------------------------------------------
    # 1. Load EDF
    # --------------------------------------------------
    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False
    )

    # --------------------------------------------------
    # 2. Extract Events
    # --------------------------------------------------
    events, event_id = mne.events_from_annotations(
    raw,
    verbose=False)

    movement_events = events[
        np.isin(
            events[:, 2],
            [event_id["T1"], event_id["T2"]]
        )
    ]

    # --------------------------------------------------
    # 3. Differential Channels
    # --------------------------------------------------
    diff_data, _ = create_differential_channels(
        raw,
        pairs
    )

    # --------------------------------------------------
    # 4. Filtering
    # --------------------------------------------------
    notch_data = apply_notch_filter(
        diff_data
    )

    filtered_data = apply_bandpass_filter(
        notch_data
    )

    # --------------------------------------------------
    # 5. Normalization
    # --------------------------------------------------
    normalized_data = minmax_normalize(
        filtered_data
    )

    # --------------------------------------------------
    # 6. Create 2-second Segments
    # --------------------------------------------------
    segment_length = 320

    segments = []
    labels = []

    for event in movement_events:

        start = event[0]
        end = start + segment_length

        if end <= normalized_data.shape[1]:

            segment = normalized_data[
                :,
                start:end
            ]

            segments.append(segment)

            labels.append(event[2])

    segments = np.array(segments)
    labels = np.array(labels)

    # --------------------------------------------------
    # 7. Convert Labels
    # T1 -> 0
    # T2 -> 1
    # --------------------------------------------------
    labels = np.where(
        labels == event_id["T1"],
        0,
        1
    )

    # --------------------------------------------------
    # 8. Create 7 Overlapping Windows
    # --------------------------------------------------
    window_size = 80
    step_size = 40

    all_windows = []

    for segment in segments:

        segment_windows = []

        for start in range(
            0,
            320 - window_size + 1,
            step_size
        ):

            end = start + window_size

            window = segment[
                :,
                start:end
            ]

            segment_windows.append(
                window
            )

        all_windows.append(
            segment_windows
        )

    all_windows = np.array(
        all_windows
    )

    # --------------------------------------------------
    # 9. Feature Extraction
    # --------------------------------------------------
    all_features = []

    for trial in all_windows:

        trial_features = []

        for window in trial:

            window_features = []

            for channel in window:

                feats = extract_11_features(
                    channel
                )

                window_features.extend(
                    feats
                )

            trial_features.append(
                window_features
            )

        all_features.append(
            trial_features
        )

    all_features = np.array(
        all_features
    )

    # --------------------------------------------------
    # Return
    # --------------------------------------------------
    return all_features, labels

In [9]:
edf = Path(
    "../data/eegmmidb/files/S001/S001R04.edf"
)

X, y = process_edf_file(edf)

print(X.shape)
print(y.shape)

(15, 7, 297)
(15,)


In [10]:
excluded_subjects = [
    43,
    88,
    89,
    92,
    100,
    104
]

valid_subjects = []

for s in range(1,110):

    if s not in excluded_subjects:
        valid_subjects.append(s)

print(len(valid_subjects))

103


In [11]:
all_X = []
all_y = []
all_subjects = []

In [12]:
for subject in valid_subjects:

    subject_id = f"S{subject:03d}"

    for run in ["R04", "R08", "R12"]:

        edf_path = Path(
            f"../data/eegmmidb/files/{subject_id}/{subject_id}{run}.edf"
        )

        if not edf_path.exists():

            print(f"Missing: {edf_path}")
            continue

        try:

            X_file, y_file = process_edf_file(
                edf_path
            )

            all_X.append(X_file)
            all_y.append(y_file)
            all_subjects.extend(
            [subject] * len(y_file)
            )

            print(
                f"{subject_id}{run} -> "
                f"{X_file.shape}"
            )

        except Exception as e:

            print(
                f"Error in {subject_id}{run}: "
                f"{e}"
            )

S001R04 -> (15, 7, 297)
S001R08 -> (15, 7, 297)
S001R12 -> (15, 7, 297)
S002R04 -> (15, 7, 297)
S002R08 -> (15, 7, 297)
S002R12 -> (15, 7, 297)
S003R04 -> (15, 7, 297)
S003R08 -> (15, 7, 297)
S003R12 -> (15, 7, 297)
S004R04 -> (15, 7, 297)
S004R08 -> (15, 7, 297)
S004R12 -> (15, 7, 297)
S005R04 -> (15, 7, 297)
S005R08 -> (15, 7, 297)
S005R12 -> (15, 7, 297)
S006R04 -> (15, 7, 297)
S006R08 -> (15, 7, 297)
S006R12 -> (15, 7, 297)
S007R04 -> (15, 7, 297)
S007R08 -> (15, 7, 297)
S007R12 -> (15, 7, 297)
S008R04 -> (15, 7, 297)
S008R08 -> (15, 7, 297)
S008R12 -> (15, 7, 297)
S009R04 -> (15, 7, 297)
S009R08 -> (15, 7, 297)
S009R12 -> (15, 7, 297)
S010R04 -> (15, 7, 297)
S010R08 -> (15, 7, 297)
S010R12 -> (15, 7, 297)
S011R04 -> (15, 7, 297)
S011R08 -> (15, 7, 297)
S011R12 -> (15, 7, 297)
S012R04 -> (15, 7, 297)
S012R08 -> (15, 7, 297)
S012R12 -> (15, 7, 297)
S013R04 -> (15, 7, 297)
S013R08 -> (15, 7, 297)
S013R12 -> (15, 7, 297)
S014R04 -> (15, 7, 297)
S014R08 -> (15, 7, 297)
S014R12 -> (15, 

In [13]:
X = np.concatenate(
    all_X,
    axis=0
)

y = np.concatenate(
    all_y,
    axis=0
)
subjects = np.array(
    all_subjects
)

print("X Shape =", X.shape)
print("y Shape =", y.shape)
print("Subjects Shape =", subjects.shape)

X Shape = (4635, 7, 297)
y Shape = (4635,)
Subjects Shape = (4635,)


In [14]:
np.save(
    "../processed/X.npy",
    X
)

np.save(
    "../processed/y.npy",
    y
)
np.save(
    "../processed/subjects.npy",
    subjects
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


In [15]:
print(X.shape)
print(y.shape)

print(np.unique(y, return_counts=True))

(4635, 7, 297)
(4635,)
(array([0, 1]), array([2337, 2298]))


In [16]:
X_loaded = np.load("../processed/X.npy")
y_loaded = np.load("../processed/y.npy")

print(X_loaded.shape)
print(y_loaded.shape)

(4635, 7, 297)
(4635,)


In [17]:
subjects = np.load(
    "../processed/subjects.npy"
)

print(subjects.shape)

print(np.unique(subjects))

print(len(np.unique(subjects)))

(4635,)
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  44  45  46  47  48  49  50  51  52  53  54  55
  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73
  74  75  76  77  78  79  80  81  82  83  84  85  86  87  90  91  93  94
  95  96  97  98  99 101 102 103 105 106 107 108 109]
103
